# Diagnóstico: base de domingos nova (SEI) vs. antiga (Pedido 097465)

Responde a três perguntas, nesta ordem:

1. A entrega antiga é equivalente à nova? (Se fosse, daria para usar a antiga e recuperar
   `zona_emb`, que na nova é `999` em ~100% das linhas.)
2. Os registros que a nova removeu eram duplicações ou embarques efetivos?
3. As repetições que **sobraram** na nova são embarques efetivos?

As duas primeiras se respondem pela estrutura interna das bases. A terceira exige fonte
externa: a série oficial de passageiros transportados, compilada por
`03_cria_base_oficial.ipynb` — **este notebook depende dele**, daí a numeração.

**Engine: pyarrow/pandas, não DuckDB.** O DuckDB 1.5.5 aborta com access violation
(`0xC0000005`) em qualquer varredura de coluna destes parquets — `count(*)` passa, porque só
lê metadados, mas `group by` e `is null` derrubam o processo, com 1, 4 ou 14 threads. O
pyarrow lê os mesmos arquivos sem erro. Como cada arquivo tem ~17 MB, agregar um a um é
barato e reproduzível.

## Seção 0 — Constantes

In [1]:
import os
import re

import pandas as pd
import pyarrow.parquet as pq

# Raiz das fontes brutas (fora do repo). Se a unidade mudar, muda so esta linha.
RAIZ_SPTRANS = r"C:\Users\9837292\Desktop\SSD\SPTrans"

PASTA_ANTIGA = os.path.join(RAIZ_SPTRANS, "Pedido 097465 - dados de bilhetagem", "arquivos PARQUET")
PASTA_NOVA = os.path.join(RAIZ_SPTRANS, "SEI! 5010.2026-0008307-9")

# Série oficial: compilada por 03_cria_base_oficial.ipynb, não recompilada aqui.
CAMINHO_OFICIAL = "../outputs/01/Dados_4Meses_Domingos_2023_2024_site.parquet"
CAMINHO_BASES = "../outputs/01/diag_bases_por_domingo.parquet"

MESES = ["04", "05", "09", "10"]          # recorte de analise; None = todos os domingos
ANOS = ["2023", "2024"]
# classes em que hash_anonimizado identifica uma pessoa. A SPTrans adverte que em
# DINHEIRO e OUTROS o mesmo hash NAO deve ser associado a um unico individuo.
CLASSES_BU = ["COMUM", "VT", "IDOSO", "PCD", "ESTUDANTE"]
CHAVE = ["linha_blt", "hash_anonimizado", "classificacao", "genero", "faixa_etaria"]

FORCE_RECOMPUTE = False
os.makedirs("../outputs/01", exist_ok=True)
pd.set_option("display.width", 200)

## Seção 1 — Inventário e integridade

Antes de qualquer comparação: os arquivos são legíveis? Esta checagem achou **3 parquets
truncados na entrega nova** (footer ausente), sem cópia local em nenhum dos ZIPs — os ZIPs
`backup/0008307-9-*.zip` contêm a versão *antiga* ("Pré e Pós"), não a revisada.
Essas datas precisam ser rebaixadas da origem.

In [2]:
def indexa(pasta):
    # Mapeia AAAAMMDD -> caminho. Os nomes da entrega antiga tem acento
    # ('Pre e Pos'), entao casamos por regex de data, nunca por nome literal.
    out = {}
    for f in os.listdir(pasta):
        if f.lower().endswith(".parquet"):
            m = re.search(r"20\d{6}", f)
            if m:
                out[m.group(0)] = os.path.join(pasta, f)
    return out


def integridade(pasta, rotulo):
    ruins = []
    for d, p in sorted(indexa(pasta).items()):
        try:
            if pq.ParquetFile(p).metadata.num_rows <= 0:
                ruins.append((d, "0 linhas"))
        except Exception as e:
            ruins.append((d, type(e).__name__))
    print(f"{rotulo}: {len(indexa(pasta))} arquivos | ilegiveis: {len(ruins)}")
    for d, m in ruins:
        print(f"   {d}  {m}")
    return {d for d, _ in ruins}


ruins_nova = integridade(PASTA_NOVA, "nova")
ruins_antiga = integridade(PASTA_ANTIGA, "antiga")

datas_comuns = sorted(set(indexa(PASTA_ANTIGA)) & set(indexa(PASTA_NOVA)))
print(f"\ndatas em ambas as entregas: {len(datas_comuns)}")
print("datas exclusivas da nova:", sorted(set(indexa(PASTA_NOVA)) - set(indexa(PASTA_ANTIGA))))
print("datas exclusivas da antiga:", sorted(set(indexa(PASTA_ANTIGA)) - set(indexa(PASTA_NOVA))))

nova: 101 arquivos | ilegiveis: 0


antiga: 101 arquivos | ilegiveis: 0

datas em ambas as entregas: 101
datas exclusivas da nova: []
datas exclusivas da antiga: []


## Seção 2 — A série oficial

Vem pronta de `03_cria_base_oficial.ipynb`, que baixa e compila os `.xls`/`.xlsx` diários
(todos os dias de 2023-2024) em `../outputs/01/Dados_4Meses_Domingos_2023_2024_site.parquet`.
Este notebook só consome — as armadilhas de parse (cabeçalho em linha variável, `Data` ora texto
ora datetime, código de linha de 6 caracteres) estão documentadas lá.

Para o diagnóstico, filtramos `tipo_dia == "domingo"`, que é o recorte das duas entregas de
bilhetagem comparadas aqui.

In [3]:
# A leitura e o parse dos .xls oficiais vivem em 03_cria_base_oficial.ipynb — aqui só consumimos.
oficial = pd.read_parquet(CAMINHO_OFICIAL)

# O diagnóstico compara domingo a domingo; a base do 03 cobre todos os dias dos dois anos.
oficial_domingo = oficial[oficial["tipo_dia"] == "domingo"]

print(f"base completa: {len(oficial):,} registros | {oficial['data'].nunique()} datas")
print(f"só domingos:   {len(oficial_domingo):,} registros | "
      f"{oficial_domingo['data'].nunique()} datas | "
      f"{oficial_domingo['linha_blt'].nunique()} linhas de ônibus")

base completa: 987,764 registros | 730 datas
só domingos:   126,403 registros | 105 datas | 1278 linhas de ônibus


### Atenção: em 2024 a quebra por modalidade colapsa — e isso é a própria política

Nos domingos de 2024 o oficial traz `of_pagantes` ~ 0 e praticamente tudo em `of_gratuidade`:
sob a Tarifa Zero ninguém paga no domingo. `of_integrados` também vai a zero, porque sem
tarifa não há integração a contabilizar.

Consequência: **a comparação por modalidade só é possível em 2023.** Entre os anos, o único
campo comparável é `of_total` (total de passageiros transportados), que continua sendo a soma
de todos os embarques nos dois regimes.

In [4]:
por_ano = oficial_domingo.groupby("Ano")[
    ["of_pagantes", "of_integrados", "of_gratuidade", "of_dinheiro", "of_total"]].sum()
print("domingos, milhões de passageiros por ano:")
print((por_ano / 1e6).round(2).to_string())

domingos, milhões de passageiros por ano:
      of_pagantes  of_integrados  of_gratuidade  of_dinheiro  of_total
Ano                                                                   
2023        56.79          24.95          23.34         6.05    109.17
2024         0.01           0.00         151.10         0.00    151.11


## Seção 3 — Agregação das duas entregas

Por domingo: registros brutos, registros com `linha_blt` preenchido, e combinações distintas
nas colunas de conteúdo (mede duplicação exata — `data` é constante dentro de cada arquivo).

In [5]:
def resume(caminho, rotulo):
    df = pq.read_table(caminho, columns=CHAVE).to_pandas()
    r = {f"{rotulo}_bruto": len(df),
         f"{rotulo}_com_linha": int(df["linha_blt"].notna().sum()),
         f"{rotulo}_distinto": int(len(df.drop_duplicates()))}
    for k, v in df.groupby("classificacao", observed=True).size().items():
        r[f"{rotulo}_cl_{k}"] = int(v)
    return r


if FORCE_RECOMPUTE or not os.path.exists(CAMINHO_BASES):
    ant, nov = indexa(PASTA_ANTIGA), indexa(PASTA_NOVA)
    linhas, descartadas = [], []
    for d in datas_comuns:
        try:
            r = {"data": d}
            r.update(resume(nov[d], "nova"))
            r.update(resume(ant[d], "antiga"))
            linhas.append(r)
        except Exception as e:
            descartadas.append((d, type(e).__name__))
    bases = pd.DataFrame(linhas).fillna(0)
    bases.to_parquet(CAMINHO_BASES, index=False)
    print("datas descartadas (arquivo ilegivel):", descartadas)
else:
    bases = pd.read_parquet(CAMINHO_BASES)

print(f"datas agregadas: {len(bases)}")
bases.head(3)[["data", "nova_bruto", "nova_com_linha", "nova_distinto",
               "antiga_bruto", "antiga_distinto"]]

datas agregadas: 97


,data,nova_bruto,nova_com_linha,nova_distinto,antiga_bruto,antiga_distinto
0,20230101,1540673,1099112,1162485,2846632,1162701
1,20230108,2601491,1966815,1909607,4649151,1909943
2,20230115,2621766,1979772,1942634,4637297,1942956


## Seção 4 — A antiga é a nova com registros duplicados

Testes sobre um domingo (2023-01-08), restritos a `CLASSES_BU` e a `linha_blt` não nulo onde
a chave exige:

| Verificação | Resultado |
|---|---|
| Usuários BU (`hash_anonimizado`) | 911.407 em ambas, conjunto idêntico |
| Linhas distintas por hash | 1,664273 nas duas — 100% dos hashes com o mesmo número |
| Conjunto de linhas por hash | 769.871/769.871 idênticos |
| Pares (hash, linha), sem linha nula | 1.516.830 / 1.516.830 — sobreposição 100,0% |
| Contagem por par | nova nunca maior que a antiga (0 casos em 1,5M) |

E o teste que fecha a questão 2: por combinação individual, `k_antiga` é múltiplo **exato**
de `k_nova` — em VT, IDOSO e PCD, *todas* as 773.049 combinações têm exatamente o dobro.
Embarques efetivos não produzem isso; é replicação mecânica.

A célula abaixo reproduz o teste para qualquer data.

In [6]:
def compara_par(data, pasta_a=PASTA_ANTIGA, pasta_b=PASTA_NOVA):
    a = pq.read_table(indexa(pasta_a)[data], columns=CHAVE).to_pandas()
    b = pq.read_table(indexa(pasta_b)[data], columns=CHAVE).to_pandas()
    a, b = [x[x["classificacao"].isin(CLASSES_BU) & x["linha_blt"].notna()] for x in (a, b)]

    ka = a.groupby(CHAVE, observed=True).size().rename("k_antiga")
    kb = b.groupby(CHAVE, observed=True).size().rename("k_nova")
    j = pd.concat([ka, kb], axis=1).dropna()

    out = j.groupby(level="classificacao", observed=True).apply(lambda g: pd.Series({
        "combos": len(g),
        "pct_1x": 100 * (g.k_antiga == g.k_nova).mean(),
        "pct_2x": 100 * (g.k_antiga == 2 * g.k_nova).mean(),
        "pct_multiplo": 100 * (g.k_antiga % g.k_nova == 0).mean(),
        "pct_nova_maior": 100 * (g.k_antiga < g.k_nova).mean(),
    }), include_groups=False)
    return out.sort_values("combos", ascending=False)


print(compara_par("20230108").to_string(float_format=lambda v: f"{v:.2f}"))

                 combos  pct_1x  pct_2x  pct_multiplo  pct_nova_maior
classificacao                                                        
COMUM         724788.00   93.59    0.04         99.99            0.00
VT            543269.00    0.00  100.00        100.00            0.00
IDOSO         152955.00    0.00  100.00        100.00            0.00
PCD            76825.00    0.00  100.00        100.00            0.00
ESTUDANTE      12463.00   94.75    0.00        100.00            0.00


## Seção 5 — Validação externa: as repetições que sobraram são embarques reais?

Comparação do total oficial de passageiros transportados contra a base nova, em três
contagens: bruta, só registros com `linha_blt`, e combinações distintas.

In [7]:
ofi_dia = oficial_domingo.groupby("data", as_index=False).agg(
    oficial_total=("of_total", "sum"), oficial_integrados=("of_integrados", "sum"))
d = ofi_dia.merge(bases, on="data").sort_values("data")
d["Ano"], d["Mes"] = d["data"].str[:4], d["data"].str[4:6]
d["nova_linha_nula"] = d["nova_bruto"] - d["nova_com_linha"]

d["r_bruto"] = d["nova_bruto"] / d["oficial_total"]
d["r_com_linha"] = d["nova_com_linha"] / d["oficial_total"]
d["r_distinto"] = d["nova_distinto"] / d["oficial_total"]
d["r_antiga"] = d["antiga_bruto"] / d["oficial_total"]

print(f"datas comparáveis: {len(d)}")
print("\n=== razao base / oficial ===")
print(d.groupby("Ano")[["r_bruto", "r_com_linha", "r_distinto", "r_antiga"]]
      .agg(["mean", "std"]).to_string(float_format=lambda v: f"{v:.3f}"))

print("\n=== registros sem linha_blt vs integrados oficiais (media por domingo) ===")
print(d.groupby("Ano")[["nova_linha_nula", "oficial_integrados"]].mean()
      .to_string(float_format=lambda v: f"{v:,.0f}"))

datas comparáveis: 93

=== razao base / oficial ===
     r_bruto       r_com_linha       r_distinto       r_antiga      
        mean   std        mean   std       mean   std     mean   std
Ano                                                                 
2023   1.397 0.099       1.035 0.038      1.040 0.096    2.843 0.316
2024   1.269 0.040       0.999 0.037      0.887 0.028    3.801 0.146

=== registros sem linha_blt vs integrados oficiais (media por domingo) ===
      nova_linha_nula  oficial_integrados
Ano                                      
2023          721,767             463,640
2024          775,306                   3


### Leitura

**`r_com_linha` = 1,035 (2023) e 0,999 (2024).** Restrita aos registros com linha, a base nova
reproduz o total oficial. `r_distinto` cai para 0,887 em 2024 — deduplicar derruba a contagem
**abaixo** do oficial.

Logo, **as repetições remanescentes na base nova são embarques efetivos**: são necessárias para
chegar ao total oficial. A resposta à pergunta 3 é sim.

**Os ~25% de registros sem `linha_blt` não são integração de ônibus.** Em 2024 o oficial
registra ~0 integrados (sem tarifa não há integração a contabilizar) e mesmo assim a base nova
mantém ~775 mil registros sem linha por domingo, praticamente o mesmo de 2023. Como o relatório
oficial cobre apenas ônibus municipais, a explicação compatível é que sejam validações em outro
modo (Metrô/CPTM com Bilhete Único). **Devem ser excluídos de qualquer contagem de demanda de
ônibus** — é o que faz a base bater com o oficial.

A base antiga fica em 2,8x (2023) e 3,8x (2024) o oficial, e o próprio fator cresce entre os
anos: é inutilizável para nível, e mesmo para variação.

## Seção 6 — Impacto na comparação interanual

2023 tem 16 domingos nos 4 meses e 2024 só 13, com composição desigual entre meses. A média
simples enviesa, então calcula-se a média por domingo dentro de cada mês e depois a média dos
4 meses.

In [8]:
q = d[d["Mes"].isin(MESES)] if MESES else d
print("=== domingos por ano-mes ===")
print(q.pivot_table(index="Mes", columns="Ano", values="data", aggfunc="count").to_string())

metricas = ["oficial_total", "nova_bruto", "nova_com_linha", "nova_distinto"]
med = q.groupby(["Ano", "Mes"])[metricas].mean().groupby("Ano").mean()
print("\n=== media por domingo, balanceada por mes ===")
print(med.to_string(float_format=lambda v: f"{v:,.0f}"))

print("\n=== crescimento 2023->2024 medido por cada fonte (%) ===")
print(((med.loc[ANOS[1]] / med.loc[ANOS[0]] - 1) * 100).round(2).to_string())

print("\n=== razao nova(com linha)/oficial por mes ===")
print(q.pivot_table(index="Mes", columns="Ano", values="r_com_linha", aggfunc="mean")
      .to_string(float_format=lambda v: f"{v:.4f}"))

=== domingos por ano-mes ===
Ano  2023  2024
Mes            
04      4     4
05      3     4
09      4     3
10      5     2

=== media por domingo, balanceada por mes ===
      oficial_total  nova_bruto  nova_com_linha  nova_distinto
Ano                                                           
2023      2,096,412   2,969,774       2,201,727      2,203,708
2024      2,850,093   3,589,659       2,817,387      2,513,179

=== crescimento 2023->2024 medido por cada fonte (%) ===
oficial_total     35.95
nova_bruto        20.87
nova_com_linha    27.96
nova_distinto     14.04

=== razao nova(com linha)/oficial por mes ===
Ano   2023   2024
Mes              
04  1.0442 0.9545
05  1.0491 1.0148
09  1.0544 0.9967
10  1.0523 0.9889


## Seção 7 — Veredito

1. **A antiga não é equivalente à nova.** É a mesma população de eventos com registros
   replicados por um fator inteiro que varia por classe de pagamento (VT e IDOSO/PCD
   exatamente 2x, COMUM ~1,12x). Não serve como fonte de nível.
2. **Os registros removidos entre as versões eram duplicação, não embarques.** O múltiplo
   inteiro exato em 100% das combinações de VT/IDOSO/PCD não é compatível com comportamento.
3. **As repetições que sobraram na nova são embarques efetivos** — validado contra a série
   oficial: com `linha_blt` preenchido, a base nova bate o total oficial (0,999 em 2024).
4. **Excluir `linha_blt` nulo é obrigatório.** São ~25% dos registros e não correspondem a
   embarques de ônibus municipal.
5. **Mesmo corrigida, a base nova subestima o crescimento do domingo** frente ao oficial
   (a razão nova/oficial cai de ~1,05 para ~0,99 entre 2023 e 2024). Qualquer conclusão sobre
   a magnitude do efeito da Tarifa Zero deve declarar isso, ou usar a série oficial para o
   nível e a bilhetagem só para a desagregação (zona, gênero, faixa etária).
6. **A entrega nova tem 3 arquivos truncados** sem cópia local. Precisam ser rebaixados.

### Sobre `zona_emb` (motivação original)

A base antiga tem `zona_emb` real e cobre 100% dos pares (hash, linha) da nova, então continua
sendo a candidata a doadora do atributo espacial — **sem** transportar suas contagens. Isso é a
Etapa 4 do plano e não foi implementado aqui.